## Authentic-only evaluation

Trains the winning grid search configuration (config 6: lr 3e-5, 60 unfrozen layers, dropout (0.3, 0.6)) under 5-fold cross-validation on the evaluation split (seed 123). The evaluation split is held out from hyperparameter selection, so the mean test accuracy and standard deviation reported here estimate generalization to data the configuration was not chosen on. 

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

import os
import json
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

Mounted at /content/drive/
TF version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Winning configuration from the grid search (config 6).
LR = 3e-5
UNFROZEN_LAYERS = 60
DROPOUT_1 = 0.3
DROPOUT_2 = 0.6

# Dots replaced with "p" so paths don't carry unexpected extensions.
CONFIG_NAME = f"winner_config6_lr{LR:.0e}_lay{UNFROZEN_LAYERS}_drop{DROPOUT_1}_{DROPOUT_2}"
CONFIG_NAME = CONFIG_NAME.replace(".", "p")

# Evaluation split (seed 123), held out from hyperparameter selection.
CV_DIR = "/content/drive/MyDrive/authentic_split_cv_eval"
RESULTS_ROOT = "/content/drive/MyDrive/authentic_training_cv_eval/model1"
CONFIG_RESULTS_DIR = f"{RESULTS_ROOT}/{CONFIG_NAME}"
RESULTS_LOG = f"{CONFIG_RESULTS_DIR}/results.json"
os.makedirs(CONFIG_RESULTS_DIR, exist_ok=True)


print(f"Checking eval split data in: {CV_DIR}")
missing_paths = []
for fold in range(5):
    fold_dir = f"{CV_DIR}/fold_{fold}"
    for split in ["train", "val", "test"]:
        path = f"{fold_dir}/{split}"
        if not os.path.exists(path):
            missing_paths.append(f"fold_{fold}/{split}")

if missing_paths:
    print(f"Missing eval split folders: {missing_paths}")
else:
    print("Eval split verified.")


if os.path.exists(RESULTS_LOG):
    with open(RESULTS_LOG, "r") as f:
        config_results = json.load(f)
    completed_folds = list(config_results.keys())
    if completed_folds:
        print(f"\nResuming completed folds: {completed_folds}")
    else:
        print("\nNo completed folds found; starting from scratch.")
else:
    config_results = {}
    completed_folds = []

print(f"\nStarting evaluation for configuration {CONFIG_NAME}")
print(f"Learning rate: {LR}, unfrozen layers: {UNFROZEN_LAYERS}, dropout: ({DROPOUT_1}, {DROPOUT_2})")

Checking eval split data in: /content/drive/MyDrive/authentic_split_cv_eval
Eval split verified.

Starting evaluation for configuration winner_config6_lr3e-05_lay60_drop0p3_0p6
Learning rate: 3e-05, unfrozen layers: 60, dropout: (0.3, 0.6)


In [ ]:
# Parameters
img_size = (682, 1024)
batch_size = 4
num_classes = 3
AUTOTUNE = tf.data.AUTOTUNE

# Train for each fold
for FOLD_IDX in range(5):
    fold_key = f"fold_{FOLD_IDX}"
    
    if fold_key in completed_folds:
        print(f"\nSkipping {fold_key} (already done)")
        continue
    
    print(f"\nStarting training for {fold_key}")
    
    # Datasets
    dataset_dir = f"{CV_DIR}/fold_{FOLD_IDX}"
    train_dir = f"{dataset_dir}/train"
    val_dir = f"{dataset_dir}/val"
    test_dir = f"{dataset_dir}/test"
    
    val_ds = keras.utils.image_dataset_from_directory(
        val_dir, labels="inferred", label_mode="int",
        image_size=img_size, batch_size=batch_size, shuffle=False,
        crop_to_aspect_ratio=True,
    )
    test_ds = keras.utils.image_dataset_from_directory(
        test_dir, labels="inferred", label_mode="int",
        image_size=img_size, batch_size=batch_size, shuffle=False,
        crop_to_aspect_ratio=True,
    )
    
    class_names = val_ds.class_names
    val_ds = val_ds.prefetch(AUTOTUNE)
    test_ds = test_ds.prefetch(AUTOTUNE)
    
    # Data augmentation
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomBrightness(0.2),
        layers.RandomContrast(0.2),
    ])
    
    t90_aug = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.10, 0.10),
        layers.RandomBrightness(0.3),
        layers.RandomContrast(0.3),
    ])
    
    def load_class_ds(class_path, label, img_size, augmentation):
        """Load a single-class folder, augment, and return (image, label) pairs."""
        ds = keras.utils.image_dataset_from_directory(
            class_path, 
            labels=None, 
            image_size=img_size,
            batch_size=None, 
            shuffle=True, 
            seed=SEED,
            crop_to_aspect_ratio=True,
        )
        ds = ds.map(lambda img: (augmentation(img, training=True), tf.cast(label, tf.int32)))
        return ds
    
    t72_ds = load_class_ds(f"{train_dir}/t72nolabel", 0, img_size, data_augmentation)
    t80_ds = load_class_ds(f"{train_dir}/t80nolabel", 1, img_size, data_augmentation)
    t90_ds = load_class_ds(f"{train_dir}/t90nolabel", 2, img_size, t90_aug)
    
    train_ds = (t72_ds
        .concatenate(t80_ds)
        .concatenate(t90_ds)
        .shuffle(buffer_size=1000, seed=SEED)
        .batch(batch_size)
        .prefetch(AUTOTUNE)
    )
    
    # Base model (ImageNet weights)
    base_model = keras.applications.EfficientNetB4(
        input_shape=img_size + (3,),
        include_top=False,
        weights="imagenet",
    )
    
    base_model.trainable = True
    for layer in base_model.layers[:-UNFROZEN_LAYERS]:
        layer.trainable = False
    
    trainable_count = sum(layer.trainable for layer in base_model.layers)
    print(f"Trainable layers in base_model: {trainable_count}/{len(base_model.layers)}")
    

    # Model
    inputs = keras.Input(shape=img_size + (3,))
    x = keras.applications.efficientnet.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT_1)(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-2))(x)
    x = layers.Dropout(DROPOUT_2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    
    model = keras.Model(inputs, outputs, name=f"m1_winner_fold{FOLD_IDX}")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"],
    )
    
    # One-hot labels
    def to_one_hot(image, label):
        return image, tf.one_hot(label, num_classes)
    
    train_ds_oh = train_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)
    val_ds_oh = val_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)
    test_ds_oh = test_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)
    
    # Class weights
    class_counts = {}
    for class_idx, cls in enumerate(class_names):
        count = len(os.listdir(f"{train_dir}/{cls}"))
        class_counts[class_idx] = count
    
    total = sum(class_counts.values())
    n_classes = len(class_counts)
    class_weight = {i: total / (n_classes * count) for i, count in class_counts.items()}
    print(f"Class weights: {class_weight}")

    # Callbacks
    CHECKPOINT_DIR = f"{CONFIG_RESULTS_DIR}/checkpoints_fold{FOLD_IDX}"
    HISTORY_PATH = f"{CONFIG_RESULTS_DIR}/history_fold{FOLD_IDX}.json"
    MODEL_PATH = f"{CONFIG_RESULTS_DIR}/model_fold{FOLD_IDX}.keras"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    
    class SaveHistoryCallback(keras.callbacks.Callback):
        """Persist the full training history to disk after every epoch.

        Colab sessions are unstable; writing a JSON snapshot after each
        epoch lets a run be inspected even if the session dies.
        """
        def __init__(self, filepath):
            self.filepath = filepath
        def on_epoch_end(self, epoch, logs=None):
            with open(self.filepath, "w") as f:
                json.dump(self.model.history.history, f)
    
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=15,
            restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=f"{CHECKPOINT_DIR}/epoch_{{epoch:02d}}_val{{val_accuracy:.3f}}.keras",
            monitor="val_accuracy",
            save_best_only=True,
            verbose=0,
        ),
        SaveHistoryCallback(filepath=HISTORY_PATH),
    ]
    
    # Train
    history = model.fit(
        train_ds_oh,
        validation_data=val_ds_oh,
        epochs=50,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=2,
    )
    
    # Evaluate
    test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
    best_val_acc = max(history.history["val_accuracy"])
    epochs_trained = len(history.history["accuracy"])
    
    print(f"\n{fold_key} results:")
    print(f"    Test accuracy: {test_acc:.4f}")
    print(f"    Best val accuracy: {best_val_acc:.4f}")
    print(f"    Epochs trained: {epochs_trained}")
    
    # Save model
    model.save(MODEL_PATH)
    
    # Update results log
    config_results[fold_key] = {
        "test_accuracy": float(test_acc),
        "test_loss": float(test_loss),
        "best_val_accuracy": float(best_val_acc),
        "epochs_trained": epochs_trained,
        "lr": LR,
        "unfrozen_layers": UNFROZEN_LAYERS,
        "dropout_1": DROPOUT_1,
        "dropout_2": DROPOUT_2,
    }
    
    with open(RESULTS_LOG, "w") as f:
        json.dump(config_results, f, indent=2)
    
    print(f"    Logged to {RESULTS_LOG}")
    print("Results saved.")
    # Release memory
    del model
    del base_model
    keras.backend.clear_session()

# Summary

print(f"=== AUTHENTIC-ONLY EVAL-SPLIT EVALUATION COMPLETE ===")

if len(config_results) == 5:
    test_accs = [config_results[f"fold_{i}"]["test_accuracy"] for i in range(5)]
    print(f"Per-fold test accuracies: {[f'{a:.4f}' for a in test_accs]}")
    print(f"Mean test accuracy: {np.mean(test_accs):.4f}")
    print(f"Std test accuracy: {np.std(test_accs):.4f}")
else:
    print(f"Completed folds: {sorted(config_results.keys())}")


Starting training for fold_0
Found 171 files belonging to 3 classes.
Found 171 files belonging to 3 classes.
Found 269 files.
Found 142 files.
Found 103 files.
71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Trainable layers in base_model: 60/475
Class weights: {0: 0.6370370370370371, 1: 1.2027972027972027, 2: 1.6699029126213591}
Epoch 1/50
129/129 - 281s - 2s/step - accuracy: 0.4553 - loss: 3.3945 - val_accuracy: 0.5322 - val_loss: 3.3141 - learning_rate: 3.0000e-05
Epoch 2/50
129/129 - 127s - 981ms/step - accuracy: 0.5097 - loss: 3.2689 - val_accuracy: 0.5965 - val_loss: 3.1610 - learning_rate: 3.0000e-05
Epoch 3/50
129/129 - 125s - 966ms/step - accuracy: 0.5798 - loss: 3.1000 - val_accuracy: 0.5556 - val_loss: 3.0617 - learning_rate: 3.0000e-05
Epoch 4/50
129/129 - 126s - 978ms/step - accuracy: 0.5856 - loss: 2.9916 - val_accuracy: 0.6374 - val_loss: 2.9302 - learning_rate: 3.0000e-05
Epoch 5/50
129/129 - 125s - 965ms/step - accuracy: 0.6381 - loss: 2.8738 - val_accuracy: 0.6257